# Практика 02. Метод ближайших соседей

**Версия:** 2026-09-25 (83871ed)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

К лекции 01. План работы:

1. **Часть 1** — kNN своими руками на NumPy: расстояния, предсказание, оценка вероятностей,
   стандартизация, кросс-валидация и метрики. Каждая функция сверяется с scikit-learn.
2. **Часть 2** — kNN из scikit-learn на реальных данных: кто из бейсболистов попал в Зал славы.
   Конвейер предобработки, подбор $k$ кросс-валидацией, итоговая оценка на тестовой выборке.
3. **Часть 3** — эксперимент и выводы: как качество зависит от $k$ и от масштабирования и почему.
   Код здесь простой, оценивается объяснение.

В части 1 решения должны быть **без циклов** по объектам — операциями над массивами (как в практике 01).
Проверка следит за этим.

In [ ]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над массивами."
    )


def make_blobs_data(n, d, n_classes, seed):
    """Синтетические данные для проверок: облака точек вокруг случайных центров."""
    r = np.random.default_rng(seed)
    centers = r.normal(scale=3, size=(n_classes, d))
    y = r.integers(0, n_classes, size=n)
    X = centers[y] + r.normal(size=(n, d))
    return X, y


print("Готово")

# Часть 1. kNN своими руками

## Задание 1.1. Матрица расстояний

Напишите `pairwise_distances(A, B, metric)` — матрицу `D` формы `(m, n)` расстояний между строками `A`
(форма `(m, d)`) и строками `B` (форма `(n, d)`) для двух метрик:

- `"euclidean"`: $\rho(\mathbf{x}, \mathbf{x}') = \left\lVert \mathbf{x} - \mathbf{x}' \right\rVert_2 = \sqrt{\sum_j (x_j - x'_j)^2}$;
- `"manhattan"`: $\rho(\mathbf{x}, \mathbf{x}') = \left\lVert \mathbf{x} - \mathbf{x}' \right\rVert_1 = \sum_j |x_j - x'_j|$.

Для любой другой метрики поднимите `ValueError`. Без циклов: вспомните задание 2.4 практики 01.
Для манхэттенской метрики подойдёт broadcasting `A[:, None, :] - B[None, :, :]`.

In [ ]:
def pairwise_distances(A, B, metric="euclidean"):
    """Матрица (m, n) расстояний между строками A и B."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.metrics import pairwise_distances as sk_pairwise_distances

A = rng.normal(size=(20, 5))
B = rng.normal(size=(7, 5))
for metric in ["euclidean", "manhattan"]:
    D = pairwise_distances(A, B, metric)
    assert D.shape == (20, 7), f"{metric}: форма должна быть (20, 7), получено {D.shape}"
    assert np.allclose(D, sk_pairwise_distances(A, B, metric=metric)), f"{metric}: расстояния не совпадают со sklearn"
assert np.allclose(np.diag(pairwise_distances(A, A)), 0, atol=1e-6), "Расстояние от точки до самой себя должно быть 0"
try:
    pairwise_distances(A, B, "cosine")
    raise AssertionError("Для неизвестной метрики должна подниматься ValueError")
except ValueError:
    pass
assert_no_loops(pairwise_distances)
print("OK")

## Задание 1.2. Предсказание kNN

Напишите две функции. Метки классов — целые числа $0, 1, \dots, K - 1$, где $K$ = `y_train.max() + 1`.

- `knn_votes(X_train, y_train, X_test, k, metric)` — матрица голосов формы `(m, K)`: для каждого
  объекта `X_test` найдите $k$ ближайших объектов обучающей выборки (`np.argsort` по строкам матрицы
  расстояний) и посчитайте, сколько среди них объектов каждого класса;
- `knn_predict(...)` — для каждого объекта класс с наибольшим числом голосов, а при ничьей —
  **меньший** номер класса (так ведёт себя `np.argmax`: он возвращает первый максимум).

Без циклов. Подсказка для `knn_votes`: `y_train[idx]` при матрице индексов `idx` формы `(m, k)` даёт
матрицу меток соседей той же формы. Её можно превратить в one-hot (практика 01, задание 2.3; здесь
удобно `np.eye(K)[labels]` — форма `(m, k, K)`) и просуммировать по оси соседей.

In [ ]:
def knn_votes(X_train, y_train, X_test, k, metric="euclidean"):
    """Матрица (m, K): сколько из k соседей каждого тестового объекта принадлежат каждому классу."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def knn_predict(X_train, y_train, X_test, k, metric="euclidean"):
    """Вектор (m,) предсказанных классов."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_tr, y_tr = make_blobs_data(300, 4, 3, seed=1)
X_te, y_te = make_blobs_data(100, 4, 3, seed=2)
votes = knn_votes(X_tr, y_tr, X_te, k=5)
assert votes.shape == (100, 3), f"Матрица голосов должна иметь форму (100, 3), получено {votes.shape}"
assert np.allclose(votes.sum(axis=1), 5), "В каждой строке голосов сумма должна быть равна k"
for k in [1, 4, 15]:
    for metric in ["euclidean", "manhattan"]:
        ours = knn_predict(X_tr, y_tr, X_te, k, metric)
        sk = KNeighborsClassifier(n_neighbors=k, metric=metric, algorithm="brute").fit(X_tr, y_tr).predict(X_te)
        assert ours.shape == (100,), f"Предсказание должно иметь форму (100,), получено {ours.shape}"
        assert np.array_equal(ours, sk), f"k={k}, {metric}: предсказания расходятся со sklearn в {np.sum(ours != sk)} объектах"
assert np.array_equal(knn_predict(X_tr, y_tr, X_tr, 1), y_tr), "При k=1 на обучающей выборке ответ должен совпадать с y_train"
assert_no_loops(knn_votes)
assert_no_loops(knn_predict)
print("OK")

## Задание 1.3. Оценка вероятностей

Оценка вероятности класса — доля соседей этого класса:
$$
\hat{p}(y \mid \mathbf{x}) = \frac{1}{k} \sum_{i=1}^{k} \mathbb{I}\left[y_{(i)} = y\right].
$$
Напишите `knn_predict_proba` — матрицу `(m, K)` таких оценок. Одна строка кода.

In [ ]:
def knn_predict_proba(X_train, y_train, X_test, k, metric="euclidean"):
    """Матрица (m, K) оценок вероятностей классов."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
P = knn_predict_proba(X_tr, y_tr, X_te, k=7)
P_sk = KNeighborsClassifier(n_neighbors=7, algorithm="brute").fit(X_tr, y_tr).predict_proba(X_te)
assert P.shape == P_sk.shape, f"Форма должна быть {P_sk.shape}, получено {P.shape}"
assert np.allclose(P.sum(axis=1), 1), "Вероятности в каждой строке должны давать в сумме 1"
assert np.allclose(P, P_sk), "Вероятности не совпадают с predict_proba из sklearn"
print("OK")

## Задание 1.4. Стандартизация как объект с fit и transform

В scikit-learn преобразования данных — объекты с методами `fit` (вычислить параметры по обучающим
данным) и `transform` (применить их к любым данным). Так легко соблюсти правило из лекции:
**параметры предобработки считаются только по обучающей выборке**.

Допишите класс `MyStandardScaler`: `fit(X)` запоминает средние `self.mean_` и стандартные отклонения
`self.scale_` столбцов (с делителем $n$, как `X.std(axis=0)`) и возвращает `self`; `transform(X)`
возвращает $(X - \mu) / \sigma$. Если у столбца $\sigma = 0$ (признак-константа), делите на 1 —
так поступает sklearn. Без циклов.

In [ ]:
class MyStandardScaler:
    def fit(self, X):
        # ВАШ КОД ЗДЕСЬ
        raise NotImplementedError

    def transform(self, X):
        # ВАШ КОД ЗДЕСЬ
        raise NotImplementedError

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [ ]:
from sklearn.preprocessing import StandardScaler

X_a = rng.normal(loc=[0, 50, 3], scale=[1, 10, 0.1], size=(100, 3))
X_a[:, 2] = 3.0  # признак-константа
X_b = rng.normal(loc=[0, 50, 3], scale=[1, 10, 0.1], size=(30, 3))
ours = MyStandardScaler()
assert ours.fit(X_a) is ours, "fit должен возвращать self"
sk = StandardScaler().fit(X_a)
assert np.allclose(ours.mean_, sk.mean_), "mean_ не совпадает со sklearn"
assert np.allclose(ours.scale_, sk.scale_), "scale_ не совпадает со sklearn (для константного столбца должно быть 1)"
assert np.allclose(ours.transform(X_b), sk.transform(X_b)), "transform новых данных не совпадает со sklearn: используйте параметры из fit"
assert not np.isnan(ours.transform(X_a)).any(), "Для константного столбца получилось деление на ноль"
assert_no_loops(MyStandardScaler.fit)
assert_no_loops(MyStandardScaler.transform)
print("OK")

## Задание 1.5. Кросс-валидация

Функция `kfold_indices` (готова) делит индексы $0, \dots, n-1$ на `n_folds` частей так же, как
`sklearn.model_selection.KFold(n_folds, shuffle=True, random_state=seed)`.

Напишите `cross_val_accuracy(X, y, k, n_folds, seed)`: для каждой части обучите kNN (то есть
возьмите ваш `knn_predict`) на остальных частях, посчитайте долю верных ответов на этой части и
верните среднее по частям. Здесь цикл по фолдам уместен: их всего несколько.

In [ ]:
def kfold_indices(n, n_folds, seed):
    """Список из n_folds массивов индексов — как в KFold(n_folds, shuffle=True, random_state=seed)."""
    perm = np.random.RandomState(seed).permutation(n)
    return np.array_split(perm, n_folds)


def cross_val_accuracy(X, y, k, n_folds=5, seed=SEED):
    """Средняя по фолдам доля верных ответов kNN."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

X_cv, y_cv = make_blobs_data(250, 3, 2, seed=3)
X_cv += rng.normal(scale=2, size=X_cv.shape)  # добавим шума, чтобы задача не была слишком простой
for k in [1, 9]:
    ours = cross_val_accuracy(X_cv, y_cv, k, n_folds=5, seed=SEED)
    sk = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, algorithm="brute"), X_cv, y_cv,
        cv=KFold(5, shuffle=True, random_state=SEED),
    ).mean()
    assert np.isclose(ours, sk), f"k={k}: ваша оценка {ours:.4f}, sklearn — {sk:.4f}"
print("OK")

## Задание 1.6. Точность, полнота, F-мера

Для бинарной задачи (положительный класс — 1) напишите `precision_recall_f1(y_true, y_pred)`,
возвращающую кортеж $(\operatorname{Precision}, \operatorname{Recall}, F_1)$:
$$
\operatorname{Precision} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}, \qquad
\operatorname{Recall} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN}}, \qquad
F_1 = \frac{2 \cdot \operatorname{Precision} \cdot \operatorname{Recall}}{\operatorname{Precision} + \operatorname{Recall}}.
$$
Если знаменатель равен нулю (например, модель не назвала положительным ни один объект), метрика
считается равной 0 — как в sklearn с `zero_division=0`. Без циклов.

In [ ]:
def precision_recall_f1(y_true, y_pred):
    """Кортеж (precision, recall, f1) для положительного класса 1."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

for trial in range(5):
    t = rng.integers(0, 2, size=50)
    p = rng.integers(0, 2, size=50)
    expected = (precision_score(t, p), recall_score(t, p), f1_score(t, p))
    assert np.allclose(precision_recall_f1(t, p), expected), f"Метрики не совпадают со sklearn: ожидалось {expected}"
t = np.array([1, 0, 1, 0])
assert precision_recall_f1(t, np.array([0, 1, 1, 0])) == (0.5, 0.5, 0.5), "Для TP=1, FP=1, FN=1 ожидается (0.5, 0.5, 0.5)"
assert precision_recall_f1(t, np.zeros(4, dtype=int)) == (0.0, 0.0, 0.0), "Если положительных предсказаний нет, все метрики равны 0"
assert_no_loops(precision_recall_f1)
print("OK")

# Часть 2. kNN в scikit-learn: Зал славы бейсбола

Данные — карьерная статистика 1340 бейсболистов, игравших в высшей лиге не меньше 10 сезонов
и завершивших карьеру до 1993 года (Cochran, *Journal of Statistics Education*, 2000; копия на OpenML,
id 185). Признаки: число сезонов и игр, выходы на биту, пробежки, хиты, хоум-раны, средний процент
отбивания, позиция на поле и т. д. Целевая переменная — попал ли игрок в **Зал славы** (1) или нет (0).
В исходных данных она принимает три значения: не член Зала славы и два способа избрания;
мы объединили оба способа в класс 1.

In [ ]:
# @title Загрузка данных: Baseball Hall of Fame (OpenML, id 185) { display-mode: "form" }
from sklearn.datasets import fetch_openml

baseball = fetch_openml(data_id=185, as_frame=True, parser="auto").frame
y_all = (baseball["Hall_of_Fame"].astype(str) != "0").astype(int).to_numpy()
X_all = baseball.drop(columns=["Hall_of_Fame"])
NUMERIC = [c for c in X_all.columns if c != "Position"]
CATEGORICAL = ["Position"]
print(f"Объектов: {len(X_all)}, признаков: {X_all.shape[1]}, доля класса 1: {y_all.mean():.3f}")
X_all.describe().T[["mean", "std", "min", "max"]]

Обратите внимание на масштабы признаков: `At_bats` измеряется тысячами, а `Batting_average` —
долями единицы. В признаке `Strikeouts` есть пропуски, `Position` — категориальный.

## Задание 2.1. Разбиение на обучающую и тестовую выборки

С помощью `train_test_split` отложите 25% данных в тестовую выборку: `random_state=SEED`,
разбиение **стратифицированное** по `y_all` (классы несбалансированы). Результат сохраните в
переменные `X_train, X_test, y_train, y_test`. Тестовую выборку дальше используем только в
задании 2.4.

In [ ]:
from sklearn.model_selection import train_test_split

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(X_test) == 335 and len(X_train) == 1005, f"Ожидалось 1005 объектов в обучении и 335 в тесте, получено {len(X_train)} и {len(X_test)}"
assert set(X_train.index).isdisjoint(X_test.index), "Обучающая и тестовая выборки пересекаются"
assert y_test.sum() == 31 and y_train.sum() == 94, (
    f"В тест должно попасть 31 из 125 членов Зала славы (стратификация), попало {y_test.sum()}: проверьте stratify=y_all"
)
print(f"OK: доля класса 1 в обучении {y_train.mean():.3f}, в тесте {y_test.mean():.3f}")

Для сравнения — качество константной модели, которая всем отвечает «не в Зале славы»:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"accuracy = {accuracy_score(y_test, dummy.predict(X_test)):.3f}, "
      f"F1 = {f1_score(y_test, dummy.predict(X_test)):.3f}")

## Задание 2.2. Конвейер

Предобработка разная для разных столбцов, поэтому соберём её из частей:

- числовые признаки: заполнить пропуски медианой (`SimpleImputer(strategy="median")`), затем, если
  `scale=True`, стандартизировать (`StandardScaler()`);
- категориальный `Position`: one-hot (`OneHotEncoder(handle_unknown="ignore")`);
- `ColumnTransformer` применяет каждую часть к своим столбцам и склеивает результат;
- `Pipeline` (или `make_pipeline`) соединяет предобработку и `KNeighborsClassifier()`.

Напишите `make_model(scale)`, возвращающую **необученный** конвейер. Благодаря конвейеру при
кросс-валидации все параметры предобработки (медианы, средние, отклонения) будут считаться только
по обучающим фолдам.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_model(scale=True):
    """Pipeline: предобработка (ColumnTransformer) + KNeighborsClassifier."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def _contains(estimator, cls):
    return cls.__name__ in repr(estimator)


for scale in [True, False]:
    model = make_model(scale)
    assert isinstance(model, Pipeline), "make_model должна возвращать Pipeline"
    assert isinstance(model.steps[-1][1], KNeighborsClassifier), "Последний шаг конвейера — KNeighborsClassifier"
    assert _contains(model, StandardScaler) == scale, f"При scale={scale} StandardScaler {'должен' if scale else 'не должен'} быть в конвейере"
    assert _contains(model, SimpleImputer) and _contains(model, OneHotEncoder), "В конвейере нужны SimpleImputer и OneHotEncoder"
    pred = model.fit(X_train, y_train).predict(X_test)
    assert pred.shape == (len(X_test),), "Конвейер должен обучаться на X_train и предсказывать для X_test"
print("OK")

## Задание 2.3. Подбор гиперпараметров

Подберите $k$ и способ взвешивания соседей поиском по сетке с кросс-валидацией:

- сетка: `n_neighbors` — нечётные числа от 1 до 51, `weights` — `"uniform"` и `"distance"`.
  Параметры шагов конвейера задаются как `"<имя шага>__<параметр>"`, например `"knn__n_neighbors"`;
- кросс-валидация: `StratifiedKFold(5, shuffle=True, random_state=SEED)`;
- метрика: `scoring="f1"` — accuracy при таком дисбалансе классов малоинформативна.

Обучите два поиска: `grid_scaled` для `make_model(scale=True)` и `grid_raw` для `make_model(scale=False)`.
Только на обучающей выборке.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

PARAM_GRID = {"knn__n_neighbors": list(range(1, 52, 2)), "knn__weights": ["uniform", "distance"]}
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("с масштабированием:", grid_scaled.best_params_, f"F1 (CV) = {grid_scaled.best_score_:.3f}")
print("без масштабирования:", grid_raw.best_params_, f"F1 (CV) = {grid_raw.best_score_:.3f}")

In [ ]:
for name, grid in [("grid_scaled", grid_scaled), ("grid_raw", grid_raw)]:
    assert isinstance(grid, GridSearchCV), f"{name} должен быть GridSearchCV"
    assert grid.scoring == "f1", f"{name}: метрика должна быть f1"
    assert len(grid.cv_results_["params"]) == 52, f"{name}: в сетке должно быть 26 значений k × 2 способа взвешивания"
    assert grid.n_splits_ == 5, f"{name}: нужна 5-кратная кросс-валидация"
    assert hasattr(grid, "best_estimator_"), f"{name}: поиск не обучен"
assert _contains(grid_scaled.best_estimator_, StandardScaler), "grid_scaled должен использовать make_model(scale=True)"
assert not _contains(grid_raw.best_estimator_, StandardScaler), "grid_raw должен использовать make_model(scale=False)"
assert grid_scaled.best_score_ > grid_raw.best_score_, "С масштабированием F1 на кросс-валидации должна быть выше — проверьте конвейеры"
print("OK")

## Задание 2.4. Итоговая оценка на тестовой выборке

Лучшая модель (`grid_scaled.best_estimator_`) уже переобучена на всей обучающей выборке. Оцените её
**один раз** на тестовой: заполните словарь `test_metrics` с ключами `"accuracy"`, `"precision"`,
`"recall"`, `"f1"`, `"roc_auc"`. Для ROC AUC нужны не классы, а оценки вероятности класса 1:
`predict_proba(X_test)[:, 1]`.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_auc_score

best = grid_scaled.best_estimator_
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print(pd.Series(test_metrics).round(3))
print("Матрица ошибок (строки — истинный класс, столбцы — предсказанный):")
print(confusion_matrix(y_test, best.predict(X_test)))

In [ ]:
assert set(test_metrics) == {"accuracy", "precision", "recall", "f1", "roc_auc"}, f"Нужны ключи accuracy, precision, recall, f1, roc_auc; получено {set(test_metrics)}"
_pred = best.predict(X_test)
assert np.isclose(test_metrics["f1"], f1_score(y_test, _pred)), "f1 посчитана неверно"
assert np.isclose(test_metrics["recall"], recall_score(y_test, _pred)), "recall посчитан неверно"
assert np.isclose(test_metrics["roc_auc"], roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])), (
    "roc_auc посчитан неверно: для него нужны вероятности predict_proba(X_test)[:, 1], а не предсказанные классы"
)
print("OK")

# Часть 3. Эксперимент и выводы

## Задание 3.1. Качество в зависимости от $k$

Результаты поиска по сетке лежат в `grid.cv_results_`: `cv_results_["params"]` — список словарей
параметров, `cv_results_["mean_test_score"]` — средняя F1 на кросс-валидации. Постройте на одном
графике зависимость средней F1 на кросс-валидации от $k$ при `weights="uniform"` для `grid_scaled`
и `grid_raw`. Удобно превратить `cv_results_` в `pd.DataFrame`.

Добавьте на тот же график F1 **на обучающей выборке** для модели с масштабированием
(`make_model(True)` с нужным `k` и `weights="uniform"`, обученной на `X_train`). Назовите результат
`train_f1` — список значений для $k = 1, 3, \dots, 51$.

In [ ]:
K_VALUES = list(range(1, 52, 2))
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(train_f1) == len(K_VALUES), f"train_f1 должен содержать {len(K_VALUES)} значений — по одному на каждое k"
assert np.isclose(train_f1[0], 1.0), "При k=1 F1 на обучающей выборке должна быть равна 1: каждый объект — сам себе ближайший сосед"
assert train_f1[-1] < train_f1[0], "При k=51 F1 на обучающей выборке должна быть ниже, чем при k=1"
print("OK")

## Задание 3.2. Выводы

Ответьте на вопросы. Опирайтесь на свои графики и числа; ответ на каждый вопрос — 2–4 предложения.

1. Какова доля класса 1 в данных и какие accuracy и F1 у константной модели? Почему для подбора $k$
   мы взяли F1, а не accuracy?
2. Как ведёт себя F1 на обучающей выборке и на кросс-валидации с ростом $k$? Объясните оба графика
   в терминах переобучения, недообучения, смещения и разброса. Почему $k$ нельзя выбирать по качеству
   на обучающей выборке?
3. Насколько масштабирование улучшило качество? Какие признаки определяют расстояние без
   масштабирования и почему это плохо для этой задачи?
4. Что было бы неправильно, если бы мы сначала применили `StandardScaler` ко всей таблице
   `X_all`, а потом разбили её и подбирали $k$? Почему это ошибка, даже если число на тесте
   почти не изменится?
5. Сравните F1 на кросс-валидации лучшей модели с F1 на тестовой выборке. Почему их не следует
   ожидать равными и почему итоговым результатом мы называем тестовое значение?
6. Предположим, модель нужна, чтобы составить короткий список кандидатов в Зал славы, которых
   потом внимательно рассмотрит комиссия, и важно не пропустить достойных. Какая метрика важнее —
   точность или полнота? Как изменить поведение обученного kNN без переобучения, чтобы её повысить?

*Ваш ответ:*